# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: <StringArray>
[ 'United States',          'China',          'India',        'Germany',
 'United Kingdom',         'France',         'Brazil',          'Japan',
         'Canada',      'Australia',    'South Korea',         'Russia',
   'South Africa',         'Mexico',      'Indonesia']
Length: 15, dtype: str

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [3]:
# --- Task 1: Multi-series line with highlight ---

highlight = 'China'
asia_df = df[df['Region'] == 'Asia'].copy()

fig = go.Figure()

# Add grey lines for all non-highlighted countries first (so highlight renders on top)
for country in asia_df['Country'].unique():
    if country == highlight:
        continue
    cdf = asia_df[asia_df['Country'] == country]
    fig.add_trace(go.Scatter(
        x=cdf['Year'], y=cdf['CO2_Mt'],
        mode='lines',
        name=country,
        line=dict(color='#DDDDDD', width=1.2),
        showlegend=False,
        hovertemplate=f'<b>{country}</b><br>Year: %{{x}}<br>CO2: %{{y:.0f}} Mt<extra></extra>'
    ))

# Add highlighted country on top
hdf = asia_df[asia_df['Country'] == highlight]
fig.add_trace(go.Scatter(
    x=hdf['Year'], y=hdf['CO2_Mt'],
    mode='lines',
    name=highlight,
    line=dict(color='#2E75B6', width=3),
    showlegend=False,
    hovertemplate=f'<b>{highlight}</b><br>Year: %{{x}}<br>CO2: %{{y:.0f}} Mt<extra></extra>'
))

# Direct label at end of highlighted line (Gestalt proximity — no legend needed)
last_row = hdf[hdf['Year'] == hdf['Year'].max()]
fig.add_annotation(
    x=last_row['Year'].values[0],
    y=last_row['CO2_Mt'].values[0],
    text=f'<b>{highlight}</b>',
    showarrow=False,
    xanchor='left', xshift=8,
    font=dict(color='#2E75B6', size=12, family='Arial')
)

fig.update_layout(
    title="China's CO2 emissions tripled since 2000 — no other Asian country comes close",
    title_font=dict(family='Arial', size=15, color='#2C2C2C'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12, color='#333333'),
    xaxis=dict(showgrid=False, title=''),
    yaxis=dict(gridcolor='#EEEEEE', title='CO2 Emissions (Mt)', zeroline=False),
    margin=dict(l=60, r=100, t=60, b=40),
    height=500
)

fig.show()


---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [4]:
# --- Task 2: Slopegraph ---

# Aggregate to regional averages for the two years
region_df = (df[df['Year'].isin([2000, 2022])]
             .groupby(['Region', 'Year'])['CO2_Mt']
             .mean()
             .reset_index())

# Determine direction of change per region (for colour coding)
pivot = region_df.pivot(index='Region', columns='Year', values='CO2_Mt')
pivot['increased'] = pivot[2022] > pivot[2000]

fig = go.Figure()

color_up   = '#2E75B6'   # blue  = increased
color_down = '#E07B00'   # amber = decreased

for region, row in pivot.iterrows():
    color = color_up if row['increased'] else color_down
    fig.add_trace(go.Scatter(
        x=[2000, 2022],
        y=[row[2000], row[2022]],
        mode='lines+markers',
        line=dict(color=color, width=2),
        marker=dict(size=7, color=color),
        showlegend=False,
        hovertemplate=f'<b>{region}</b><br>%{{x}}: %{{y:.1f}} Mt<extra></extra>'
    ))

    # Left label (2000 value)
    fig.add_annotation(
        x=2000, y=row[2000],
        text=f'{row[2000]:.0f}',
        showarrow=False, xanchor='right', xshift=-10,
        font=dict(color=color, size=10, family='Arial')
    )
    # Right label: region name + 2022 value
    fig.add_annotation(
        x=2022, y=row[2022],
        text=f'{region}<br><b>{row[2022]:.0f} Mt</b>',
        showarrow=False, xanchor='left', xshift=8,
        font=dict(color=color, size=10, family='Arial'),
        align='left'
    )

fig.update_layout(
    title='Asia surged; North America and Europe cut emissions — the regional gap widened sharply',
    title_font=dict(family='Arial', size=14, color='#2C2C2C'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12, color='#333333'),
    xaxis=dict(
        tickvals=[2000, 2022],
        ticktext=['2000', '2022'],
        showgrid=False,
        range=[1994, 2034],   # extra room for labels on both sides
        title=''
    ),
    yaxis=dict(
        showgrid=False,
        showticklabels=False,  # endpoint labels make tick labels redundant
        title=''
    ),
    margin=dict(l=80, r=180, t=60, b=40),
    height=550,
    width=900
)

fig.show()